# D334 — Apache Sqoop Introduction

Apache Sqoop is a command-line bulk-transfer tool between relational databases and the Hadoop ecosystem. Use it here on **Amazon EMR 6.15.0** to move data between **Amazon RDS for MySQL** and HDFS.

Focus on the legacy-tool workflow: the data path, run one import, and run one export. Cluster and RDS creation remain outside the notebook; assume both are in the same default VPC and can communicate on MySQL port 3306.

## 1. Why Sqoop existed

Moving a large table one row at a time through a custom script is slow and difficult to parallelize. Sqoop reads database metadata, generates a MapReduce job, divides a table into ranges, and lets mapper tasks transfer those ranges concurrently.

```text
Import: RDS MySQL table -> JDBC -> Sqoop/MapReduce -> HDFS
Export: HDFS files      -> Sqoop/MapReduce -> JDBC -> existing MySQL table
```

Sqoop is a **batch transfer tool**, not streaming ingestion, change-data capture, a query engine, or a database replication system.

## 2. Project status: legacy technology

Sqoop 1.4.7 was released in **December 2017** and no later Apache Sqoop release followed. It is reasonable to say active release development effectively stopped in 2017. Apache formally retired Sqoop in **June 2021** and moved it to the Apache Attic in July 2021.

Amazon EMR 6.15.0 still contains Sqoop 1.4.7, and supports this workflow. AWS later removed Sqoop starting with EMR 7.5. For new production platforms, evaluate maintained JDBC ingestion, Spark JDBC, AWS Database Migration Service, AWS Glue, or a CDC tool according to the use case.

Sqoop remains relevant when supporting older Hadoop/EMR pipelines and demonstrates parallel database-to-Hadoop movement.

## 3. Import and export

### Import

`sqoop import` reads an RDBMS table or query and writes records to HDFS. Common controls include:

- `--connect`, `--username`, and password handling;
- `--table` or `--query`;
- `--columns` and `--where`;
- `--target-dir`;
- `--num-mappers` / `-m` and `--split-by`;
- output formats such as delimited text, Avro, or SequenceFile;
- incremental modes using an increasing/check column.

### Export

`sqoop export` reads HDFS files and writes rows to an **already-created** database table. The file delimiter, column order, types, null representation, keys, and constraints must agree with the destination schema. Export can insert or update, but it is not a transactional database-wide synchronization mechanism.

## 4. Parallelism and the split column

With multiple mappers, Sqoop usually finds minimum and maximum values of a split column, divides the range, and gives each mapper a predicate. A numeric, indexed, reasonably uniform primary key is a good split column.

```text
Mapper 1: customer_id 1–25000
Mapper 2: customer_id 25001–50000
Mapper 3: customer_id 50001–75000
Mapper 4: customer_id 75001–100000
```

Skew creates slow mappers. Too many mappers create too many concurrent database connections and can overload RDS. Our tiny lab deliberately uses `-m 1`; this needs no split column and makes the transfer easy to inspect. Production parallelism must be sized with the database owner.

## 5. What happens to schema and types?

Sqoop inspects JDBC metadata and maps SQL types to Java/Hadoop representations. Some database-specific types, unsigned numbers, decimals, dates/timestamps, binary values, and nulls need explicit testing. Delimited text does not carry a strong schema by itself.

An import can create a Hive table with additional options, but this lab intentionally lands plain CSV-like text in HDFS so the transfer remains visible. On export, Sqoop does **not** create the MySQL target table: create it first with compatible columns in the same order, or provide `--columns` explicitly.

## 6. EMR and RDS assumptions

For the practical notebook:

- EMR release is `emr-6.15.0`, with Sqoop 1.4.7 installed as an application.
- RDS engine is MySQL, master username is `admin`, and the database password is supplied through a protected password file.
- RDS and EMR are in the same default VPC, and connectivity to the RDS endpoint on port 3306 is already allowed.
- Commands run from the EMR primary node as user `hadoop`; MapReduce tasks run through YARN.
- Use the RDS **endpoint hostname**, never `localhost`. Mapper containers must be able to resolve and reach that endpoint.

AWS documents that EMR installs the MariaDB JDBC driver by default. MariaDB Connector/J can communicate with RDS MySQL, so the commands use `jdbc:mariadb://...` with `org.mariadb.jdbc.Driver` and requires no driver download.

## 7. Reliability boundaries

- A failed import normally leaves output that should be inspected or removed before retrying; Sqoop refuses an import when the target directory already exists.
- Export mappers commit independently. A failed job can leave some rows committed, so a blind rerun may encounter duplicate keys.
- Source rows may change during a long import; Sqoop alone does not guarantee a consistent whole-database snapshot.
- Incremental import tracks an append key or last-modified timestamp, but job state, late data, updates, and deletes require deliberate handling.
- Plain command-line passwords can appear in process listings and logs. Use prompting, a protected password file, or an appropriate credential mechanism.

Use deterministic tables and cleanup to make behavior reproducible.

## 8. Command map

| Command | Purpose |
|---|---|
| `sqoop version` | Confirm installation |
| `sqoop list-databases` | Test JDBC access and list databases |
| `sqoop list-tables` | List tables in one database |
| `sqoop eval` | Run a small SQL statement/query through JDBC |
| `sqoop import` | RDBMS to HDFS/Hive/HBase |
| `sqoop export` | HDFS to an existing RDBMS table |
| `sqoop job` | Save/manage Sqoop job definitions |
| `sqoop metastore` | Shared metadata service for saved jobs |

`eval` is convenient for diagnostics, but Sqoop is not intended as an interactive SQL client.

## 9. Verification questions

1. Which direction does `sqoop import` move data?
2. Why does a four-mapper import need a suitable split column?
3. Does `sqoop export` create its MySQL target table?
4. Why must the JDBC URL use the RDS endpoint instead of `localhost`?
5. Is Sqoop a recommended new CDC platform?

<details><summary>Answers</summary>

1. From the relational database into Hadoop storage.  
2. To divide rows into balanced, non-overlapping mapper ranges.  
3. No.  
4. Distributed tasks must reach the remote database; their localhost is not RDS.  
5. No; it is retired batch-transfer technology, not CDC.
</details>

## 10. References and next notebook

Continue with **D335_SqoopPractical.ipynb**.

Official references:

- [Sqoop on Amazon EMR](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-sqoop.html)
- [EMR Sqoop considerations and JDBC drivers](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-sqoop-considerations.html)
- [Apache Sqoop 1.4.7 User Guide](https://sqoop.apache.org/docs/1.4.7/SqoopUserGuide.html)
- [Apache Attic: Sqoop project status](https://attic.apache.org/projects/sqoop.html)